# Joint-Truncation SVD Merge — Isolating Budget Size from Truncation Method

The previous adaptive-rank-merge (independent per-adapter SVD truncation, then sum) regressed hard: GSM8K collapsed to 0.0000, worse than every existing method including DARE. Diagnostic completions showed COHERENT but confidently WRONG reasoning — not degenerate garbage — consistent with genuine loss of task-specific signal from truncating each adapter independently before combination, on adapters that are largely non-redundant (~88-89 degrees apart).

This notebook isolates the two conflated variables from that experiment:
- **Truncation method**: joint (truncate the COMBINED matrix, same as the existing `svd_merge` — already confirmed gentle, 95% energy retained at rank=16) instead of independent (truncate each adapter separately, then sum — the likely culprit).
- **Budget size**: sweep several values above 16, motivated by the effective-rank finding that metamath/codealpaca use ~89-90% of their nominal rank-16 already, making a shared budget of only 16 for three adapters tight.

rank=48 is included as a sanity check — with no truncation loss (rank ≥ actual combined rank), it should behave close to plain linear merge.

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth peft
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"


## Load Adapters + Base Model

In [2]:
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download
from unsloth import FastLanguageModel
import json
import torch

def get_lora_deltas(repo_name: str, hf_username: str = "Srishtik", lora_alpha: int = None, r: int = None) -> dict:
    repo_id = f"{hf_username}/{repo_name}"
    config_path = hf_hub_download(repo_id=repo_id, filename="adapter_config.json")
    cfg = json.load(open(config_path))
    actual_r     = r if r is not None else cfg.get("r")
    actual_alpha = lora_alpha if lora_alpha is not None else cfg.get("lora_alpha")
    path = hf_hub_download(repo_id=repo_id, filename="adapter_model.safetensors")
    adapter_weights = load_file(path)
    scale = actual_alpha / actual_r
    layers = {}
    for key, val in adapter_weights.items():
        if "lora_A" in key:
            base_key = key.replace("lora_A.default.weight", "").replace("lora_A.weight", "")
            layers.setdefault(base_key, {})["A"] = val.float()
        elif "lora_B" in key:
            base_key = key.replace("lora_B.default.weight", "").replace("lora_B.weight", "")
            layers.setdefault(base_key, {})["B"] = val.float()
    deltas = {}
    for base_key, mats in layers.items():
        if "A" in mats and "B" in mats:
            deltas[base_key] = scale * (mats["B"] @ mats["A"])
    print(f"  {repo_name:<35} r={actual_r}  alpha={actual_alpha}  scale={scale:.4f}")
    return deltas

print("Loading adapter deltas:")
dolly_deltas      = get_lora_deltas("qwen3-trained-on-dolly-15k")
metamath_deltas   = get_lora_deltas("qwen3-trained-on-metamath-15k")
codealpaca_deltas = get_lora_deltas("qwen3-trained-on-code-alpaca-18k")

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen3-0.6B",
    max_seq_length = 2048,
    load_in_4bit   = False,
    dtype          = torch.float16,
)
base_sd = base_model.state_dict()
del base_model
torch.cuda.empty_cache()

def normalize_delta_keys(deltas: dict) -> dict:
    normalized = {}
    for k, v in deltas.items():
        new_key = k.replace("base_model.model.", "").rstrip(".") + ".weight"
        normalized[new_key] = v
    return normalized

dolly_deltas      = normalize_delta_keys(dolly_deltas)
metamath_deltas   = normalize_delta_keys(metamath_deltas)
codealpaca_deltas = normalize_delta_keys(codealpaca_deltas)
deltas = [dolly_deltas, metamath_deltas, codealpaca_deltas]
adapter_names = ["dolly", "metamath", "codealpaca"]

print(f"\nOverlap with base_sd: {len(set(dolly_deltas.keys()) & set(base_sd.keys()))}")  # expect 196

def apply_delta_to_base(base_state_dict: dict, delta_state_dict: dict) -> dict:
    merged = {}
    for key, base_val in base_state_dict.items():
        if key in delta_state_dict:
            delta_val = delta_state_dict[key].to(base_val.device)
            merged[key] = (base_val.float() + delta_val).to(base_val.dtype)
        else:
            merged[key] = base_val
    return merged


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading adapter deltas:


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/40.4M [00:00<?, ?B/s]

  qwen3-trained-on-dolly-15k          r=16  alpha=32  scale=2.0000


adapter_config.json: 0.00B [00:00, ?B/s]

...

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/40.4M [00:00<?, ?B/s]

  qwen3-trained-on-code-alpaca-18k    r=16  alpha=32  scale=2.0000
==((====))==  Unsloth 2026.8.2: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]


Overlap with base_sd: 196


## Joint SVD Merge (Budget Is the Only Variable)

In [3]:
import torch

def joint_svd_merge(deltas: list, rank: int, weights: list = None) -> dict:
    """
    Same construction as the original svd_merge: sum the deltas FIRST, then
    SVD-truncate the COMBINED matrix to `rank`. This is deliberately NOT the
    adaptive-rank-merge approach (which truncated each adapter independently
    before summing, and regressed hard: GSM8K 0.00 vs linear's 0.11).

    Isolates one variable: does a LARGER rank budget (motivated by the
    measured effective-rank finding — metamath/codealpaca use ~89-90% of
    their rank-16 budget, so a shared budget of only 16 for three adapters is
    tight) help, while keeping the JOINT truncation that already proved gentle
    (95% energy retained at rank=16 on the combined matrix, vs independent
    per-adapter truncation's apparent much larger, more destructive loss).
    """
    n = len(deltas)
    weights = weights or [1.0 / n] * n
    keys = set.intersection(*[set(d.keys()) for d in deltas])
    merged = {}
    for key in keys:
        combined = sum(w * d[key] for w, d in zip(weights, deltas))
        U, S, Vh = torch.linalg.svd(combined.float(), full_matrices=False)
        r = min(rank, len(S))
        merged[key] = U[:, :r] @ torch.diag(S[:r]) @ Vh[:r, :]
    return merged


## Eval Functions (needed before the sweep, which evaluates as it goes)

In [4]:
import torch
import gc
import re
import math
import multiprocessing
import contextlib
import io
from datasets import load_dataset
from tqdm import tqdm
from unsloth import FastLanguageModel

def build_chat_prompt(tokenizer, user_content: str) -> str:
    messages = [{"role": "user", "content": user_content}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )

def prep_tokenizer_for_generation(tokenizer):
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

# ── GSM8K ──
def extract_gsm8k_answer(text: str) -> str:
    hash_match = re.findall(r"####\s*(-?[\d,]+\.?\d*)", text)
    if hash_match:
        return hash_match[-1].replace(",", "").strip()
    boxed = re.findall(r"\\boxed\{([^}]+)\}", text)
    if boxed:
        return boxed[-1].replace(",", "").strip()
    numbers = re.findall(r"-?\d[\d,]*\.?\d*", text)
    if numbers:
        return numbers[-1].replace(",", "").strip()
    return text.strip()

def format_gsm8k_prompt(question: str) -> str:
    return (f"Solve the following math problem. Show your reasoning and put "
            f"your final numeric answer after '#### '.\n\nQuestion: {question}")

def evaluate_gsm8k(model, tokenizer, model_name="model", num_samples=200, batch_size=4,
                    max_new_tokens=320, device="cuda"):
    print(f"\n{'─'*60}\n[GSM8K] Evaluating: {model_name}\n{'─'*60}")
    model.eval()
    prep_tokenizer_for_generation(tokenizer)
    dataset = load_dataset("gsm8k", "main", split="test")
    dataset = dataset.select(range(min(num_samples, len(dataset))))
    preds, labels = [], []
    for i in tqdm(range(0, len(dataset), batch_size), desc=f"{model_name} [gsm8k]"):
        batch = dataset[i : i + batch_size]
        questions    = batch["question"]
        true_answers = [extract_gsm8k_answer(a) for a in batch["answer"]]
        prompts      = [build_chat_prompt(tokenizer, format_gsm8k_prompt(q)) for q in questions]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                            max_length=512, add_special_tokens=False).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                      pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
                                      repetition_penalty=1.3)
        input_len = inputs["input_ids"].shape[1]
        for j, output in enumerate(outputs):
            generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            preds.append(extract_gsm8k_answer(generated))
            labels.append(true_answers[j])
    per_sample_exact = [int(p.strip() == l.strip()) for p, l in zip(preds, labels)]
    exact_match = round(sum(per_sample_exact) / len(per_sample_exact), 4)
    print(f"  Exact Match: {exact_match:.4f}")
    return {"repo_id": model_name, "exact_match": exact_match, "num_samples": len(per_sample_exact),
            "per_sample_exact": per_sample_exact}

# ── HumanEval ──
def extract_code(generated: str, problem_prompt: str, entry_point: str) -> str:
    text = generated.strip()
    fence = re.search(r"```(?:python)?\s*\n?(.*?)```", text, re.DOTALL)
    if fence:
        text = fence.group(1).strip()
    if f"def {entry_point}" in text:
        return text
    return problem_prompt + "\n" + text

def _unsafe_execute(program: str, result_list, timeout: int):
    import signal
    def handler(signum, frame):
        raise TimeoutError("execution timed out")
    try:
        signal.signal(signal.SIGALRM, handler)
        signal.alarm(timeout)
        exec_globals = {}
        with contextlib.redirect_stdout(io.StringIO()):
            exec(program, exec_globals)
        signal.alarm(0)
        result_list.append("passed")
    except Exception as e:
        result_list.append(f"failed: {type(e).__name__}: {e}")

def check_correctness(problem: dict, completion_code: str, timeout: int = 5) -> bool:
    program = completion_code + "\n" + problem["test"] + f"\ncheck({problem['entry_point']})\n"
    manager = multiprocessing.Manager()
    result_list = manager.list()
    p = multiprocessing.Process(target=_unsafe_execute, args=(program, result_list, timeout))
    p.start()
    p.join(timeout=timeout + 1)
    if p.is_alive():
        p.kill(); p.join()
    if not result_list:
        result_list.append("failed: timeout")
    return result_list[0] == "passed"

def format_humaneval_prompt(problem_prompt: str) -> str:
    return ("Complete the following Python function. Return ONLY the complete "
            "function code (including the signature), with no explanations and "
            f"no markdown formatting.\n\n{problem_prompt}")

def evaluate_humaneval(model, tokenizer, model_name="model", num_samples=164, batch_size=4,
                        max_new_tokens=384, device="cuda"):
    print(f"\n{'─'*60}\n[HumanEval] Evaluating: {model_name}\n{'─'*60}")
    model.eval()
    prep_tokenizer_for_generation(tokenizer)
    dataset = load_dataset("openai/openai_humaneval", split="test")
    dataset = dataset.select(range(min(num_samples, len(dataset))))
    per_sample_pass = []
    for i in tqdm(range(0, len(dataset), batch_size), desc=f"{model_name} [humaneval]"):
        batch = dataset[i : i + batch_size]
        problem_prompts = batch["prompt"]
        entry_points    = batch["entry_point"]
        prompts = [build_chat_prompt(tokenizer, format_humaneval_prompt(p)) for p in problem_prompts]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                            max_length=768, add_special_tokens=False).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                      pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
                                      repetition_penalty=1.1)
        input_len = inputs["input_ids"].shape[1]
        for j, output in enumerate(outputs):
            generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            code = extract_code(generated, problem_prompts[j], entry_points[j])
            problem = {"prompt": problem_prompts[j], "test": batch["test"][j], "entry_point": entry_points[j]}
            per_sample_pass.append(int(check_correctness(problem, code, timeout=5)))
    pass_at_1 = round(sum(per_sample_pass) / len(per_sample_pass), 4)
    print(f"  pass@1: {pass_at_1:.4f}")
    return {"repo_id": model_name, "pass_at_1": pass_at_1, "num_samples": len(per_sample_pass),
            "per_sample_pass": per_sample_pass}

# ── Dolly-15k perplexity ──
def format_dolly_prompt(instruction: str, context: str) -> str:
    if context:
        return f"Instruction: {instruction}\nContext: {context}\nResponse:"
    return f"Instruction: {instruction}\nResponse:"

def evaluate_dolly_perplexity(model, tokenizer, model_name="model", num_samples=200,
                               max_length=512, device="cuda"):
    print(f"\n{'─'*60}\n[Dolly-PPL] Evaluating: {model_name}\n{'─'*60}")
    model.eval()
    dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
    dataset = dataset.shuffle(seed=42).select(range(min(num_samples, len(dataset))))
    per_sample_nll, per_sample_ppl = [], []
    for ex in tqdm(dataset, desc=f"{model_name} [dolly-ppl]"):
        prompt = format_dolly_prompt(ex["instruction"], ex.get("context", ""))
        response = ex["response"]
        if not response.strip():
            continue
        full_text = prompt + " " + response
        prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length).input_ids.to(device)
        full_ids   = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=max_length).input_ids.to(device)
        if full_ids.shape[1] <= prompt_ids.shape[1]:
            continue
        labels = full_ids.clone()
        labels[:, : prompt_ids.shape[1]] = -100
        with torch.no_grad():
            out = model(full_ids, labels=labels)
        nll = out.loss.item()
        per_sample_nll.append(nll)
        per_sample_ppl.append(math.exp(nll))
    result = {"repo_id": model_name, "perplexity": round(sum(per_sample_ppl) / len(per_sample_ppl), 4),
              "mean_nll": round(sum(per_sample_nll) / len(per_sample_nll), 4),
              "num_samples": len(per_sample_ppl), "per_sample_nll": per_sample_nll}
    print(f"  Perplexity: {result['perplexity']:.4f}  (mean NLL: {result['mean_nll']:.4f})")
    return result


## Budget Sweep (n=60 proxy — larger than the previous n=40 sweep, still a proxy)

**Fixed:** the quick-eval builder now loads the base model with `load_in_4bit=False` before loading the merged state dict — a 4-bit-quantized model's weights are packed into a flattened blob shape that a normal float state_dict can't be loaded into.

In [5]:
import gc
import numpy as np

def build_and_evaluate_budget(rank, gsm8k_samples=60):
    merged_delta = joint_svd_merge(deltas, rank=rank)
    merged_sd    = apply_delta_to_base(base_sd, merged_delta)

    from unsloth import FastLanguageModel
    # load_in_4bit=False here — a 4-bit model's weights are packed into a
    # flattened quantized blob shape (e.g. [1572864, 1]), which a normal
    # float state_dict cannot be loaded into. Quantize AFTER loading if
    # inference speed matters; for a 60-sample proxy this isn't necessary.
    model, _ = FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen3-0.6B", max_seq_length=1024, load_in_4bit=False, dtype=torch.float16,
    )
    target_dtype = next(model.parameters()).dtype
    cast_sd = {k: v.to(target_dtype) if v.is_floating_point() else v for k, v in merged_sd.items()}
    missing, unexpected = model.load_state_dict(cast_sd, strict=False)
    if missing or unexpected:
        print(f"  [warn] missing={len(missing)} unexpected={len(unexpected)}")
    FastLanguageModel.for_inference(model)

    result = evaluate_gsm8k(model, tokenizer, model_name=f"joint-svd-budget-{rank}",
                             num_samples=gsm8k_samples, batch_size=4)

    del model, cast_sd
    gc.collect()
    torch.cuda.empty_cache()
    return result["exact_match"]


BUDGETS = [16, 20, 24, 32, 40, 48]  # 48 = sanity check, should behave ~linear (no truncation loss)
sweep_results = {}

for rank in BUDGETS:
    em = build_and_evaluate_budget(rank, gsm8k_samples=60)
    sweep_results[rank] = em
    print(f"rank={rank:<4} GSM8K EM (n=60 proxy) = {em:.4f}")

print(f"\n{'='*50}")
print("Summary (GSM8K EM, n=60 proxy — still a proxy, confirm the winner at n=200):")
for rank, em in sweep_results.items():
    print(f"  rank={rank:<4} EM={em:.4f}")

best_rank = max(sweep_results, key=sweep_results.get)
print(f"\nBest candidate: rank={best_rank} (EM={sweep_results[best_rank]:.4f} at n=60)")
print("Reference — linear (n=200, already known): EM=0.1100")


==((====))==  Unsloth 2026.8.2: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────
[GSM8K] Evaluating: joint-svd-budget-16
────────────────────────────────────────────────────────────


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

...

  Exact Match: 0.1333
rank=40   GSM8K EM (n=60 proxy) = 0.1333
==((====))==  Unsloth 2026.8.2: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────
[GSM8K] Evaluating: joint-svd-budget-48
────────────────────────────────────────────────────────────


joint-svd-budget-48 [gsm8k]: 100%|██████████| 15/15 [01:43<00:00,  6.91s/it]


  Exact Match: 0.1333
rank=48   GSM8K EM (n=60 proxy) = 0.1333

Summary (GSM8K EM, n=60 proxy — still a proxy, confirm the winner at n=200):
  rank=16   EM=0.1167
  rank=20   EM=0.2333
  rank=24   EM=0.2167
  rank=32   EM=0.1333
  rank=40   EM=0.1333
  rank=48   EM=0.1333

Best candidate: rank=20 (EM=0.2333 at n=60)
Reference — linear (n=200, already known): EM=0.1100


## Upload the Best-Scoring Budget

In [ ]:
import os
import torch
from unsloth import FastLanguageModel

merged_delta_best = joint_svd_merge(deltas, rank=best_rank)
merged_sd_best = apply_delta_to_base(base_sd, merged_delta_best)
print(f"Best-budget merged state dict ready (rank={best_rank}): {len(merged_sd_best)} keys")


def upload_merged_model(merged_sd, repo_name, tokenizer, hf_token,
                         base_repo="unsloth/Qwen3-0.6B", max_seq_length=2048,
                         dtype=torch.float16, push_to_hub=True):
    print(f"[upload] Preparing model → {repo_name}")
    model, _ = FastLanguageModel.from_pretrained(
        model_name=base_repo, max_seq_length=max_seq_length, load_in_4bit=False, dtype=dtype,
    )
    target_dtype = next(model.parameters()).dtype
    cast_sd = {k: v.to(target_dtype) if v.is_floating_point() else v for k, v in merged_sd.items()}
    missing, unexpected = model.load_state_dict(cast_sd, strict=False)
    if missing:
        print(f"  [warn] Missing keys   : {len(missing)}  (e.g. {missing[:3]})")
    if unexpected:
        print(f"  [warn] Unexpected keys: {len(unexpected)} (e.g. {unexpected[:3]})")
    model.eval()
    if push_to_hub:
        commit_info = model.push_to_hub(repo_name, token=hf_token, private=False)
        tokenizer.push_to_hub(repo_name, token=hf_token, private=False)
        print(f"  [hub] Pushed → https://huggingface.co/{repo_name}")
        print(f"  [hub] Commit info: {commit_info}")
    del model, cast_sd
    torch.cuda.empty_cache()
    return repo_name


# Use Kaggle Secrets rather than pasting a token directly:
# from kaggle_secrets import UserSecretsClient
# HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
HF_TOKEN = key

JOINT_SVD_REPO = f"Srishtik/Qwen3-0.6B-joint-svd-rank{best_rank}-3-adapters-merged-2"
uploaded_joint_svd_repo = upload_merged_model(
    merged_sd = merged_sd_best,
    repo_name = JOINT_SVD_REPO,
    tokenizer = tokenizer,
    hf_token  = HF_TOKEN,
)


Best-budget merged state dict ready (rank=20): 311 keys
[upload] Preparing model → Srishtik/Qwen3-0.6B-joint-svd-rank20-3-adapters-merged-2
==((====))==  Unsloth 2026.8.2: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/526 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Saved model to https://huggingface.co/Srishtik/Qwen3-0.6B-joint-svd-rank20-3-adapters-merged-2


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp2m78ua9n/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  [hub] Pushed → https://huggingface.co/Srishtik/Qwen3-0.6B-joint-svd-rank20-3-adapters-merged-2
  [hub] Commit info: None


## Full Evaluation (n=200/164/200) + Comparison

In [7]:
import gc

eval_model, eval_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = uploaded_joint_svd_repo,
    max_seq_length = 1024,
    load_in_4bit   = True,
    dtype          = torch.float16,
)
FastLanguageModel.for_inference(eval_model)

joint_gsm8k     = evaluate_gsm8k(eval_model, eval_tokenizer, model_name=uploaded_joint_svd_repo,
                                  num_samples=200, batch_size=4)
joint_humaneval = evaluate_humaneval(eval_model, eval_tokenizer, model_name=uploaded_joint_svd_repo,
                                      num_samples=164, batch_size=4)
joint_dolly     = evaluate_dolly_perplexity(eval_model, eval_tokenizer, model_name=uploaded_joint_svd_repo,
                                             num_samples=200)

del eval_model, eval_tokenizer
gc.collect()
torch.cuda.empty_cache()

known_results = {
    "linear"              : {"gsm8k": 0.1100, "humaneval": 0.2012, "dolly_ppl": 17.8096},
    "svd (rank=16)"        : {"gsm8k": 0.1550, "humaneval": 0.2073, "dolly_ppl": 20.5070},
    "ties"                 : {"gsm8k": 0.0550, "humaneval": 0.2073, "dolly_ppl": 16.7428},
    "dare"                 : {"gsm8k": 0.0400, "humaneval": 0.2378, "dolly_ppl": 17.2781},
    "adaptive-rank (independent trunc, FAILED)": {"gsm8k": 0.0000, "humaneval": 0.1585, "dolly_ppl": 19.1178},
    "codealpaca_adapter (specialist)": {"gsm8k": 0.0850, "humaneval": 0.1707, "dolly_ppl": 38.3633},
    "metamath_adapter (specialist)"  : {"gsm8k": 0.2200, "humaneval": 0.1220, "dolly_ppl": 28.6676},
    "dolly_adapter (specialist)"     : {"gsm8k": 0.0250, "humaneval": 0.1646, "dolly_ppl": 12.6278},
}

print(f"\n{'═'*72}")
print(f"{'Model':<42}{'GSM8K':>10}{'HumanEval':>10}{'Dolly PPL':>10}")
print("-" * 72)
for name, r in known_results.items():
    print(f"{name:<42}{r['gsm8k']:>10.4f}{r['humaneval']:>10.4f}{r['dolly_ppl']:>10.4f}")
print("-" * 72)
print(f"{'joint-svd-rank' + str(best_rank) + ' (NEW)':<42}{joint_gsm8k['exact_match']:>10.4f}"
      f"{joint_humaneval['pass_at_1']:>10.4f}{joint_dolly['perplexity']:>10.4f}")
print("=" * 72)

if joint_gsm8k['exact_match'] > known_results["svd (rank=16)"]["gsm8k"]:
    print(f"\nLarger joint-truncation budget BEAT rank-16 SVD on GSM8K "
          f"({joint_gsm8k['exact_match']:.4f} vs {known_results['svd (rank=16)']['gsm8k']:.4f}) "
          f"— the effective-rank finding helps when combined with JOINT (not independent) truncation.")
else:
    print(f"\nLarger budget did NOT beat rank-16 SVD on GSM8K "
          f"({joint_gsm8k['exact_match']:.4f} vs {known_results['svd (rank=16)']['gsm8k']:.4f}) "
          f"— rank-16 SVD may already be close to sufficient; the earlier failure was likely "
          f"specific to independent-then-sum truncation, not budget size.")


==((====))==  Unsloth 2026.8.2: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────
[GSM8K] Evaluating: Srishtik/Qwen3-0.6B-joint-svd-rank20-3-adapters-merged-2
────────────────────────────────────────────────────────────


Srishtik/Qwen3-0.6B-joint-svd-rank20-3-adapters-merged-2 [gsm8k]: 100%|██████████| 50/50 [09:03<00:00, 10.86s/it]

  Exact Match: 0.1250

────────────────────────────────────────────────────────────
[HumanEval] Evaluating: Srishtik/Qwen3-0.6B-joint-svd-rank20-3-adapters-merged-2
────────────────────────────────────────────────────────────


...

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Srishtik/Qwen3-0.6B-joint-svd-rank20-3-adapters-merged-2 [dolly-ppl]: 100%|██████████| 200/200 [00:26<00:00,  7.55it/s]


  Perplexity: 19.4030  (mean NLL: 2.0700)

════════════════════════════════════════════════════════════════════════
Model                                          GSM8K HumanEval Dolly PPL
------------------------------------------------------------------------
linear                                        0.1100    0.2012   17.8096
svd (rank=16)                                 0.1550    0.2073   20.5070
ties                                          0.0550    0.2073   16.7428
dare                                          0.0400    0.2378   17.2781
adaptive-rank (independent trunc, FAILED)     0.0000    0.1585   19.1178
codealpaca_adapter (specialist)               0.0850    0.1707   38.3633
metamath_adapter (specialist)                 0.2200    0.1220   28.6676
dolly_adapter (specialist)                    0.0250    0.1646   12.6278
------------------------------------------------------------------------
joint-svd-rank20 (NEW)                        0.1250    0.1829   19.4030

Larger 